# CMIP6 core recipe example

Shows the recipe API with the bundled `cmip6.core_units` recipe.

Flow: load recipe -> check -> dry-run -> apply -> re-check.

In [ ]:
import numpy as np

import woodpecker
from woodpecker.testing import make_cmip6

Create a CMIP6-like dataset where `tas` is stored in Celsius instead of Kelvin.

In [ ]:
dataset = make_cmip6(overrides={"units": "degC"}, seed=7)
original_values = dataset["tas"].values.copy()

dataset

Load the recipe and inspect the selected core units fix.

In [ ]:
recipe = woodpecker.recipe.get("cmip6.core_units")

recipe.model_dump()

In [ ]:
findings = woodpecker.recipe.check(dataset, recipe)

findings.fix_ids

Dry-run previews the repair without changing the dataset.

In [ ]:
result = woodpecker.recipe.apply(dataset, recipe, dry_run=True)

(
    result.stats,
    result.preview,
    dataset["tas"].attrs["units"],
    np.allclose(dataset["tas"].values, original_values),
)

Apply the recipe in memory and re-check.

In [ ]:
write = woodpecker.recipe.apply(dataset, recipe, dry_run=False)

(
    write.stats,
    dataset["tas"].attrs["units"],
    np.allclose(dataset["tas"].values, original_values + 273.15),
)

In [ ]:
recheck = woodpecker.recipe.check(dataset, recipe)

bool(recheck)